# 999 · Port a Python de `aede` y `clahfac` (decisión D6)

**Objetivo.** Verificar que `src/camiz_port.py`, la traducción a Python de `R/camiz_port.R`
(métodos y código original del Dr. Sergio Camiz, AedeClahfac 1.2), reproduce los resultados de
la versión en R, que es la referencia (D6).

**Criterio de aceptación.**

1. Los CSV `R_*` de las dos vías, escritos por `R/tesis_ciclo_financiero.Rmd`
   (`data/results/niveles/R_*.csv`, vía sin transformar, D1; `data/results/R_*.csv`, vía
   transformada, D2), se reproducen con tolerancia absoluta de $10^{-10}$ en las cifras y con
   coincidencia exacta en los cortes, los $k$ elegidos y la jerarquía (comparada por código de
   serie del BCRP).
2. Comparación complementaria contra `R/camiz_port.R` ejecutado en este entorno (salidas
   intermedias con 17 cifras significativas), si `Rscript` está disponible.

**Salidas.** `data/results/999_port_paridad.csv` (prueba, vía, diferencia máxima, estado) y las
tablas del port por vía: `999_port_aede_cortes.csv`, `999_port_hfc_jerarquia.csv`,
`999_port_ehfc_cofenetica.csv` (vía transformada en `data/results/`, vía sin transformar en
`data/results/niveles/`).

**Semillas.** Ningún cálculo de este notebook es aleatorio (EPCA, Fisher, HFC y PCA son
determinísticos); no se fija semilla. Se declara `SEMILLA = None` por trazabilidad.

**Sustituciones declaradas.** La HFC es la del asesor (port de `clahfac`), no `hclust`/`scipy`.
No existe un algoritmo EHFC publicado: lo que aquí se llama «HFC evolutiva» es la aplicación de
`clahfac` por ventanas, tal como la hace el Rmd, y no se presenta como método publicado.


In [1]:
# ---- Configuración ----------------------------------------------------------
import hashlib
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

RAIZ = Path.cwd().resolve()
if not (RAIZ / '_quarto.yml').exists():
    RAIZ = RAIZ.parent
assert (RAIZ / '_quarto.yml').exists(), 'No se encontró la raíz del repositorio'
sys.path.insert(0, str(RAIZ))
from src import camiz_port as cp

SEMILLA = None                 # sin componentes aleatorios
TOL = 1e-10                    # tolerancia absoluta para cifras (criterio de aceptación)
VENTANA, PASO, MAXCL, NDIM = 60, 3, 10, 3   # mismos parámetros que el Rmd (b4-replica, b4-aede)
DIR_T = RAIZ / 'data' / 'results'           # vía transformada (D2)
DIR_N = DIR_T / 'niveles'                   # vía sin transformar (D1, oficial)
print('numpy', np.__version__, '· pandas', pd.__version__, '· Python', sys.version.split()[0])


numpy 2.5.3 · pandas 3.0.5 · Python 3.13.16


## 1. Datos: verificación antes de modelar

Se verifican la versión congelada (D4, md5), el calendario mensual continuo, la ausencia de
faltantes y las dimensiones (228 × 17 en niveles; 227 × 17 transformadas). Las 17 series y su
orden son los de `meta_an` en el Rmd (`entra == "Sí"`): se excluyen el crédito total del SF y el
IPC subyacente. Nombres cortos y códigos BCRP copiados de `metadatos_series()` del Rmd.

Los CSV se leen con `cp.leer_csv_como_r`, que reproduce el conversor de texto a número de R
(`R_strtod`), para que Python y R partan de los mismos bits. La sección 7.1 mide qué cambia si se
usa el lector de pandas.

In [2]:
# ---- Metadatos (orden de meta_an en el Rmd) ------------------------------------
META = pd.DataFrame([
    ('Crédito empresarial', 'Crédito empresarial', 'PN00532MM'),
    ('Crédito consumo', 'Crédito consumo', 'PN00533MM'),
    ('Crédito hipotecario', 'Crédito hipotecario', 'PN00534MM'),
    ('Liquidez M1', 'M1 (dinero)', 'PN00199MM'),
    ('Liquidez M2', 'M2 (liquidez en soles)', 'PN00208MM'),
    ('Liquidez M3', 'M3 (liquidez total)', 'PN00214MM'),
    ('Tasa referencia BCRP', 'Tasa de referencia', 'PD04722MM'),
    ('Tasa interbancaria', 'Tasa interbancaria', 'PN07819NM'),
    ('Tasa activa TAMN', 'TAMN', 'PN07807NM'),
    ('Tasa pasiva TIPMN', 'TIPMN', 'PN07816NM'),
    ('IPC Lima', 'IPC Lima', 'PN38705PM'),
    ('Tipo de cambio', 'Tipo de cambio', 'PN01246PM'),
    ('Índice BVL', 'Índice BVL', 'PN01142MM'),
    ('embi_peru', 'EMBIG Perú', 'PD04709XD'),
    ('PBI desestacionalizado', 'PBI desestacionalizado', 'PN01773AM'),
    ('Demanda interna', 'Demanda interna', 'PN01774AM'),
    ('Reservas internacionales', 'Reservas internacionales netas', 'PN00027MM'),
], columns=['columna', 'nombre_corto', 'codigo'])
P = len(META)

MD5_D4 = {'dataset_niveles_mensual.csv': '8088c28cf277419388d9562722faa1ad',
          'dataset_transformado_mensual.csv': '8c49f3e55608f2651be3c1dee4236194'}

def md5(ruta):
    return hashlib.md5(Path(ruta).read_bytes()).hexdigest()

def verificar(d, n_esperado, inicio, fin):
    # Calendario mensual continuo, sin faltantes y con la dimensión declarada
    f = pd.to_datetime(d['fecha'])
    cal = pd.date_range(inicio, fin, freq='MS')
    assert len(f) == len(cal) and (f.to_numpy() == cal.to_numpy()).all(), 'calendario con huecos'
    assert not d.isna().any().any(), 'hay valores faltantes'
    assert len(d) == n_esperado

for nombre, h in MD5_D4.items():
    assert md5(RAIZ / 'data' / 'processed' / nombre) == h, f'{nombre} no es la versión congelada (D4)'

# Se leen los CSV con los mismos bits que read.csv de R (cp.leer_csv_como_r; ver sección 7.1)
niv = cp.leer_csv_como_r(RAIZ / 'data/processed/dataset_niveles_mensual.csv')
tr = cp.leer_csv_como_r(RAIZ / 'data/processed/dataset_transformado_mensual.csv')
verificar(niv, 228, '2007-01-01', '2025-12-01')
verificar(tr, 227, '2007-02-01', '2025-12-01')

X = niv[META.columna].copy()
X.columns = META.nombre_corto
X.index = niv['fecha'].str[:7]
base = {c.replace('_log_diff', '').replace('_diff', ''): c for c in tr.columns if c != 'fecha'}
Xt = tr[[base[c] for c in META.columna]].copy()
Xt.columns = META.nombre_corto
Xt.index = tr['fecha'].str[:7]
assert X.shape == (228, 17) and Xt.shape == (227, 17)
VIAS = {'niveles': (X, DIR_N), 'transformadas': (Xt, DIR_T)}
print('niveles', X.shape, X.index[0], '→', X.index[-1], '| transformadas', Xt.shape, Xt.index[0], '→', Xt.index[-1])


niveles (228, 17) 2007-01 → 2025-12 | transformadas (227, 17) 2007-02 → 2025-12


## 2. Registro de las pruebas

Cada prueba agrega una fila con: grupo (`aceptación` = CSV `R_*`; `complementaria` = sección 7),
prueba, vía, archivo de referencia, rutina de valores propios usada, tipo de comparación
(`exacta` para enteros, fechas y etiquetas; `tolerancia` para cifras), diferencia máxima y
estado (`coincide` / `difiere`). La diferencia de una prueba exacta es el número de elementos
que no coinciden.

In [3]:
registro = []

MOTOR = 'numpy'   # rutina de valores propios en uso (ver sección 7.2)

def anotar(prueba, via, referencia, tipo, dif, detalle='', grupo='aceptación'):
    ok = (dif == 0) if tipo == 'exacta' else bool(np.isfinite(dif) and dif <= TOL)
    registro.append({'grupo': grupo, 'prueba': prueba, 'via': via, 'referencia': referencia,
                     'motor_valores_propios': MOTOR, 'tipo': tipo,
                     'tolerancia': 0 if tipo == 'exacta' else TOL, 'dif_max': float(dif),
                     'estado': 'coincide' if ok else 'difiere', 'explicacion': '', 'detalle': detalle})
    return ok

def dif_num(a, b):
    # Diferencia absoluta máxima; los NA deben estar en las mismas posiciones
    a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)
    if a.shape != b.shape or (np.isnan(a) != np.isnan(b)).any():
        return np.inf
    m = ~np.isnan(a)
    return float(np.max(np.abs(a[m] - b[m]))) if m.any() else 0.0

def rel(ruta):
    return str(Path(ruta).relative_to(RAIZ))


## 3. EPCA del asesor (`aede`): k elegidos y cortes de Fisher

Como en el bloque `b4-aede` (y `bB3-aede`): ventanas de 60 meses con paso 1, tres dimensiones,
pesos uniformes y gaussianos (`nsd = 3`), Fisher hasta 10 clases sobre la serie del primer
valor propio. Tres criterios: codo (caída máxima de la reducción relativa), Calinski-Harabász con
$k$ grados de libertad (fórmula del código de `aede`) y con $k-1$ (forma habitual). Se compara el
CSV `R_aede_cortes.csv` de cada vía: $k$ y fechas de corte, exactos.

In [4]:
CRITERIOS = [('codo (caída máxima)', 'k_codo', 'cortes_codo'),
             ('Calinski-Harabász k gl (aede)', 'k_ch', 'cortes_ch'),
             ('Calinski-Harabász k-1 gl (habitual)', 'k_ch1', 'cortes_ch1')]
aede_py = {}
for via, (M, d) in VIAS.items():
    filas = []
    for nsd, pesos in [(0, 'uniformes'), (3, 'gaussianos_nsd3')]:
        a = cp.aede(M, nw=VENTANA, ndim=NDIM, nsd=nsd, maxcl=MAXCL)
        r = cp.resumen_aede(a)
        aede_py[(via, nsd)] = (a, r)
        for crit, kk, cc in CRITERIOS:
            filas.append({'pesos': pesos, 'criterio': crit, 'k': r[kk], 'cortes': ' '.join(r[cc])})
    py = pd.DataFrame(filas)
    py.to_csv(d / '999_port_aede_cortes.csv', index=False)
    ref = pd.read_csv(d / 'R_aede_cortes.csv')
    assert list(ref.pesos) == list(py.pesos) and list(ref.criterio) == list(py.criterio)
    anotar('aede: k elegido (6 criterios)', via, rel(d / 'R_aede_cortes.csv'), 'exacta',
           int((ref.k.to_numpy() != py.k.to_numpy()).sum()), 'k: ' + ' '.join(map(str, py.k)))
    anotar('aede: fechas de corte (6 criterios)', via, rel(d / 'R_aede_cortes.csv'), 'exacta',
           int((ref.cortes.to_numpy() != py.cortes.to_numpy()).sum()))
    print(via); display(py)


niveles


,pesos,criterio,k,cortes
0,uniformes,codo (caída máxima),2,2020-03
1,uniformes,Calinski-Harabász k gl (aede),10,2012-08 2013-06 2014-04 2015-05 2017-03 2019-1...
2,uniformes,Calinski-Harabász k-1 gl (habitual),2,2020-03
3,gaussianos_nsd3,codo (caída máxima),2,2020-05
4,gaussianos_nsd3,Calinski-Harabász k gl (aede),10,2012-05 2013-05 2014-10 2015-07 2017-02 2020-0...
5,gaussianos_nsd3,Calinski-Harabász k-1 gl (habitual),10,2012-05 2013-05 2014-10 2015-07 2017-02 2020-0...


transformadas


,pesos,criterio,k,cortes
0,uniformes,codo (caída máxima),4,2011-12 2017-10 2019-11
1,uniformes,Calinski-Harabász k gl (aede),10,2010-08 2011-05 2011-12 2013-11 2017-10 2019-1...
2,uniformes,Calinski-Harabász k-1 gl (habitual),10,2010-08 2011-05 2011-12 2013-11 2017-10 2019-1...
3,gaussianos_nsd3,codo (caída máxima),4,2011-05 2018-08 2020-02
4,gaussianos_nsd3,Calinski-Harabász k gl (aede),10,2010-11 2011-11 2013-11 2016-09 2018-05 2018-1...
5,gaussianos_nsd3,Calinski-Harabász k-1 gl (habitual),10,2010-11 2011-11 2013-11 2016-09 2018-05 2018-1...


## 4. HFC (`clahfac`): jerarquía, valores propios por nodo e índices

Se compara `R_hfc_jerarquia.csv`: los nodos unidos (`N1`, `N2`) por **código de serie** para las
hojas y por número para los nodos, las series de cada hoja, y las 13 columnas numéricas de la
jerarquía (`Num`, pesos, `Eigenv` = $\lambda_1$, `Index` = $\lambda_2$, porcentajes locales y
globales, `Eigentab`, `Ratio`, coeficientes de los hijos) con tolerancia $10^{-10}$.

In [5]:
COLS_NUM = ['ngr', 'Num', 'Weight', 'Weight2', 'Eigenv', 'Index', 'Loc_%1', 'Loc_%2', 'Glob_%1',
            'Glob_%2', 'Cum_%2', 'Eigentab', 'Ratio', 'Weight_N1', 'Weight_N2']
hfc_py = {}

def jerarquia_con_series(h):
    # Igual que jerarquia_con_series() del Rmd: identifica las hojas por nombre y código BCRP
    j = h['hierar'].copy()
    et = lambda k: META.nombre_corto[k - 1] if k <= P else f'Nodo {k}'
    co = lambda k: META.codigo[k - 1] if k <= P else np.nan
    pos = list(j.columns).index('N2') + 1
    extra = pd.DataFrame({'N1_serie': j.N1.map(et), 'N1_codigo': j.N1.map(co),
                          'N2_serie': j.N2.map(et), 'N2_codigo': j.N2.map(co)}, index=j.index)
    return pd.concat([j.iloc[:, :pos], extra, j.iloc[:, pos:]], axis=1)

for via, (M, d) in VIAS.items():
    h = cp.clahfac(M)
    hfc_py[via] = h
    py = jerarquia_con_series(h)
    py.to_csv(d / '999_port_hfc_jerarquia.csv')
    ref = pd.read_csv(d / 'R_hfc_jerarquia.csv', index_col=0)
    arch = rel(d / 'R_hfc_jerarquia.csv')
    assert list(ref.index) == list(py.index)
    # Jerarquía por código: hojas por código BCRP, nodos internos por número
    clave = lambda df, c: [df[c + '_codigo'].iloc[i] if isinstance(df[c + '_codigo'].iloc[i], str)
                           else f'Nodo {df[c].iloc[i]}' for i in range(len(df))]
    n_dif = sum(a != b for c in ['N1', 'N2'] for a, b in zip(clave(ref, c), clave(py, c)))
    anotar('HFC: nodos unidos N1/N2 (por código de serie)', via, arch, 'exacta', n_dif)
    n_dif = sum((ref[c].astype(str).to_numpy() != py[c].astype(str).to_numpy()).sum() for c in ['N1_serie', 'N2_serie'])
    anotar('HFC: nombres de las series unidas', via, arch, 'exacta', int(n_dif))
    # Composición de cada nodo (conjunto de códigos), a partir de nclass
    comp = lambda k: frozenset(META.codigo[v - 1] for v in h['nclass'].loc[k] if v > 0)
    anotar('HFC: tamaño de cada nodo (Num)', via, arch, 'exacta', int((ref.Num != py.Num).sum()))
    for col_grupo, cols in [('λ1 e índice (Eigenv, Index)', ['Eigenv', 'Index']),
                            ('porcentajes locales y globales', ['Loc_%1', 'Loc_%2', 'Glob_%1', 'Glob_%2', 'Cum_%2']),
                            ('Eigentab y Ratio', ['Eigentab', 'Ratio']),
                            ('pesos del nodo y de sus hijos', ['Weight', 'Weight2', 'Weight_N1', 'Weight_N2'])]:
        anotar(f'HFC: {col_grupo}', via, arch, 'tolerancia', dif_num(ref[cols], py[cols]))
    print(via, '· raíz: λ1 =', round(py.Eigenv.iloc[-1], 6), '· índice =', round(py.Index.iloc[-1], 6),
          '· grupos de la raíz:', len(comp(int(py.N1.iloc[-1]))) if py.N1.iloc[-1] > P else 1, 'y',
          len(comp(int(py.N2.iloc[-1]))) if py.N2.iloc[-1] > P else 1, 'series')


niveles · raíz: λ1 = 11.319422 · índice = 2.76445 · grupos de la raíz: 14 y 3 series
transformadas · raíz: λ1 = 3.788987 · índice = 2.449931 · grupos de la raíz: 11 y 6 series


## 5. HFC por ventanas: correlación cofenética y Rand ajustado

Como en `b5-ehfc` / `bB4-ehfc`: `clahfac` en cada ventana de 60 meses con paso 3; entre
ventanas consecutivas, la correlación cofenética de Pearson (equivalente a
`dendextend::cor_cophenetic` sobre `as.dendrogram`) y el índice de Rand ajustado entre las
particiones en 4 grupos de `.partclas_ac`. Fechas (centro de ventana) exactas; cifras con
tolerancia $10^{-10}$.

In [6]:
ehfc_py = {}
for via, (M, d) in VIAS.items():
    ini = list(range(0, len(M) - VENTANA + 1, PASO))
    jers = [cp.clahfac(M.iloc[i:i + VENTANA])['hier'] for i in ini]
    fechas = [M.index[i + VENTANA // 2] for i in ini]
    cof = [cp.cor_cofenetica(jers[i], jers[i - 1]) for i in range(1, len(jers))]
    grupos = [cp.partclas(h, 4)['partition']['Position'].to_numpy() for h in jers]
    ari = [cp.rand_ajustado(grupos[i], grupos[i - 1]) for i in range(1, len(grupos))]
    py = pd.DataFrame({'fecha': fechas[1:], 'cofenetica': cof, 'rand_ajustado_4g': ari})
    py.to_csv(d / '999_port_ehfc_cofenetica.csv', index=False)
    ehfc_py[via] = jers
    ref = pd.read_csv(d / 'R_ehfc_cofenetica.csv')
    arch = rel(d / 'R_ehfc_cofenetica.csv')
    anotar('HFC por ventanas: fechas (centro de ventana)', via, arch, 'exacta',
           int(len(ref) != len(py)) or int((ref.fecha.to_numpy() != py.fecha.to_numpy()).sum()), f'{len(py)} comparaciones')
    anotar('HFC por ventanas: correlación cofenética', via, arch, 'tolerancia', dif_num(ref.cofenetica, py.cofenetica))
    anotar('HFC por ventanas: Rand ajustado (4 grupos)', via, arch, 'tolerancia', dif_num(ref.rand_ajustado_4g, py.rand_ajustado_4g))
    print(via, len(jers), 'ventanas · cofenética mín.', round(min(cof), 4), '· Rand ajustado mediano', round(float(np.median(ari)), 3))


niveles 57 ventanas · cofenética mín. 0.2391 · Rand ajustado mediano 0.956


transformadas 56 ventanas · cofenética mín. 0.2636 · Rand ajustado mediano 0.697


## 6. Criterios de la datación PCA-MM12 (`R_regimenes_criterios.csv`)

Este CSV no sale de `aede` ni de `clahfac`, sino de `datar()` del Rmd (bloques `b8-datacion` y
`bB6-datacion`): PCA con la regla de signo de Rcamiz, media móvil causal de 12 meses de PC1–PC3,
Fisher multivariado con segmentos de al menos 12 meses y Calinski-Harabász en sus dos variantes
(`ch_variantes`, sí portada en `src/camiz_port.py`). Se reproduce aquí con funciones locales de
**solo validación**, traducidas del Rmd; las funciones oficiales de Python para la datación son
responsabilidad de `src/particion.py` (otro desarrollador) y no se usan ni se modifican.

In [7]:
# ---- Funciones de validación traducidas del Rmd (no reutilizables) ----------
def estandarizar(M):
    M = np.asarray(M, dtype=float)
    m = np.sum(M.astype(np.longdouble), axis=0).astype(float) / len(M)
    c = M - m
    s = np.sqrt(np.sum((c * c).astype(np.longdouble), axis=0).astype(float) / len(M))
    return c / s

def pca_basico(Z):
    # Regla de signo de Rcamiz: domina la suma de cuadrados de los coeficientes positivos
    val, V = cp._eigen_sim(cp._crossprod_ref(Z) / len(Z))
    V = V.copy()
    for k in range(V.shape[1]):
        v = V[:, k]
        if np.sum(v[v < 0] ** 2) > np.sum(v[v > 0] ** 2):
            V[:, k] = -v
    return val, V, Z @ V

def mm_causal(x, w):
    # stats::filter(x, rep(1/w, w), sides = 1): suma en el orden x[t], x[t-1], ...
    r = np.full(len(x), np.nan)
    for t in range(w - 1, len(x)):
        z = 0.0
        for j in range(w):
            z += (1 / w) * x[t - j]
        r[t] = z
    return r

def particion_fisher(Y, k_max=10, tam_min=12):
    Y = np.asarray(Y, dtype=float); n = len(Y)
    S1 = np.vstack([np.zeros(Y.shape[1]), np.cumsum(Y.astype(np.longdouble), axis=0).astype(float)])
    S2 = np.concatenate([[0.0], np.cumsum(np.sum((Y * Y).astype(np.longdouble), axis=1)).astype(float)])
    def sce(i, j):
        s = S1[j] - S1[i]
        return S2[j] - S2[i] - float(np.sum((s * s).astype(np.longdouble))) / (j - i)
    D = np.full((k_max + 1, n + 1), np.inf); B = np.zeros((k_max + 1, n + 1), dtype=int)
    for j in range(tam_min, n + 1):
        D[1, j] = sce(0, j)
    for k in range(2, k_max + 1):
        if k * tam_min > n:
            break
        for j in range(k * tam_min, n + 1):
            ini = np.arange((k - 1) * tam_min, j - tam_min + 1)
            c = np.array([D[k - 1, i] + sce(i, j) for i in ini])
            q = int(np.argmin(c)); D[k, j] = c[q]; B[k, j] = ini[q]
    res = {}
    for k in range(1, k_max + 1):
        if not np.isfinite(D[k, n]):
            continue
        cortes = []; j = n
        for kk in range(k, 1, -1):
            i = B[kk, j]; cortes.insert(0, i); j = i
        res[k] = {'sce': D[k, n], 'cortes': cortes}
    return res

def datar_tabla(S):
    res = particion_fisher(estandarizar(S), 10, 12)
    ks = np.array(sorted(res)); sce = np.array([res[k]['sce'] for k in ks])
    red = np.append(-np.diff(sce) / sce[:-1], np.nan)
    ch = cp.ch_variantes(sce, len(S))
    return pd.DataFrame({'k': ks, 'sce': sce, 'reduccion': red, 'ch': ch['k'], 'ch1': ch['k1']})

L = 12
for via, (M, d) in VIAS.items():
    val, V, S = pca_basico(estandarizar(M.to_numpy()))
    mm = np.column_stack([mm_causal(S[:, k], L) for k in range(3)])[L - 1:]
    py = datar_tabla(mm)
    ref = pd.read_csv(d / 'R_regimenes_criterios.csv')
    arch = rel(d / 'R_regimenes_criterios.csv')
    anotar('Datación PCA-MM12: grilla de k', via, arch, 'exacta', int(len(ref) != len(py)) or int((ref.k != py.k).sum()))
    anotar('Datación PCA-MM12: SC intra y reducción', via, arch, 'tolerancia', dif_num(ref[['sce', 'reduccion']], py[['sce', 'reduccion']]))
    anotar('Datación PCA-MM12: Calinski-Harabász k y k-1 gl', via, arch, 'tolerancia', dif_num(ref[['ch', 'ch1']], py[['ch', 'ch1']]))
    k_ch, k_ch1 = int(py.k[np.nanargmax(py.ch)]), int(py.k[np.nanargmax(py.ch1)])
    anotar('Datación PCA-MM12: k de Calinski-Harabász (k y k-1 gl)', via, arch, 'exacta',
           int(k_ch != int(ref.k[np.nanargmax(ref.ch)])) + int(k_ch1 != int(ref.k[np.nanargmax(ref.ch1)])),
           f'k gl: {k_ch}; k-1 gl: {k_ch1}')


## 7. Pruebas complementarias

### 7.1 Sensibilidad al lector de CSV

`read.csv` de R convierte el texto a número con `R_strtod`, que acumula los dígitos en doble
extendida y divide por la potencia de 10 antes de redondear a doble precisión. Ese doble redondeo
hace que algunos valores queden a 1 ulp del valor correctamente redondeado que da pandas
(`float_precision='round_trip'`). Se cuenta cuántos valores cambian y se repiten las pruebas de
aceptación principales con el lector de pandas.

In [8]:
lect = {}
for nombre, Mr in [('dataset_niveles_mensual.csv', X), ('dataset_transformado_mensual.csv', Xt)]:
    pdv = pd.read_csv(RAIZ / 'data/processed' / nombre, float_precision='round_trip')
    rv = cp.leer_csv_como_r(RAIZ / 'data/processed' / nombre)
    c = [k for k in pdv.columns if k != 'fecha']
    dif = pdv[c].to_numpy() != rv[c].to_numpy().astype(float)
    print(nombre, '· valores distintos entre lectores:', int(dif.sum()), 'de', dif.size,
          [(pdv.fecha[i], c[j], repr(pdv[c[j]][i]), repr(float(rv[c[j]][i]))) for i, j in zip(*np.nonzero(dif))])
    lect[nombre] = pdv

Xp = lect['dataset_niveles_mensual.csv'][META.columna].copy(); Xp.columns = META.nombre_corto; Xp.index = X.index
trp = lect['dataset_transformado_mensual.csv']
Xtp = trp[[base[c] for c in META.columna]].copy(); Xtp.columns = META.nombre_corto; Xtp.index = Xt.index
for via, (Mp, d) in {'niveles': (Xp, DIR_N), 'transformadas': (Xtp, DIR_T)}.items():
    ref_a = pd.read_csv(d / 'R_aede_cortes.csv')
    n_dif = 0
    for nsd, pesos in [(0, 'uniformes'), (3, 'gaussianos_nsd3')]:
        r = cp.resumen_aede(cp.aede(Mp, nw=VENTANA, ndim=NDIM, nsd=nsd, maxcl=MAXCL))
        for crit, kk, cc in CRITERIOS:
            f = ref_a[(ref_a.pesos == pesos) & (ref_a.criterio == crit)].iloc[0]
            n_dif += int(f.k != r[kk]) + int(f.cortes != ' '.join(r[cc]))
    anotar('Lector pandas · aede: k y cortes', via, rel(d / 'R_aede_cortes.csv'), 'exacta', n_dif, grupo='complementaria')
    jp = jerarquia_con_series(cp.clahfac(Mp)); jr = pd.read_csv(d / 'R_hfc_jerarquia.csv', index_col=0)
    anotar('Lector pandas · HFC: N1/N2', via, rel(d / 'R_hfc_jerarquia.csv'), 'exacta',
           int((jp[['N1', 'N2']].to_numpy() != jr[['N1', 'N2']].to_numpy()).sum()), grupo='complementaria')
    cols_num = [c for c in COLS_NUM if c != 'ngr']
    anotar('Lector pandas · HFC: cifras de la jerarquía', via, rel(d / 'R_hfc_jerarquia.csv'), 'tolerancia',
           dif_num(jp[cols_num], jr[cols_num]), grupo='complementaria')


dataset_niveles_mensual.csv · valores distintos entre lectores: 1 de 4332 [('2018-09-01', 'Liquidez M1', 'np.float64(85281.551109)', '85281.55110899999')]
dataset_transformado_mensual.csv · valores distintos entre lectores: 0 de 3859 []


### 7.2 Contra `R/camiz_port.R` ejecutado en este entorno

Los CSV `R_*` traen 15 cifras significativas y no incluyen todas las salidas. Si `Rscript` está
disponible, se ejecuta `R/camiz_port.R` (sin tocar el Rmd; el script se escribe en un directorio
temporal) y se vuelcan con 17 cifras: valores propios por ventana, tabla de Fisher completa,
correlaciones evolutivas (`correlationsgn`, `correlationlsq`) y diferencias de mínimos cuadrados
de `aede`; y de `clahfac`: jerarquía completa, factores de nodo (`repvar`, `datout`), `corout`,
`correp`, inercias, `nclass`, `npart`, `nsg`, `noeuind`, orden del dendrograma, `Eigentab`
convergido, `clahpart` (k = 1..4: clases, ítems y coordenadas), `.partclas_ac`, `.hc2axes_ac` y
las jerarquías de todas las ventanas.

La comparación se hace dos veces: con la rutina de valores propios por defecto del port
(`numpy.linalg.eigh`, LAPACK `dsyevd` sobre OpenBLAS) y con `dsyevr` del LAPACK de referencia
del sistema, que es la que llama `eigen()` de R en este entorno (`cp.activar_lapack_referencia`).
Si una diferencia de la primera desaparece en la segunda, su causa es la rutina de valores
propios y no el port.

In [9]:
R_VOLCADO = r"""
args <- commandArgs(trailingOnly = TRUE); raiz <- args[1]; sal <- args[2]
suppressWarnings(Sys.setlocale("LC_CTYPE", "C.UTF-8")); setwd(raiz)
source("R/camiz_port.R")
cols <- strsplit(args[3], "|", fixed = TRUE)[[1]]; nombres <- strsplit(args[4], "|", fixed = TRUE)[[1]]
niv <- read.csv("data/processed/dataset_niveles_mensual.csv", check.names = FALSE, encoding = "UTF-8")
tr <- read.csv("data/processed/dataset_transformado_mensual.csv", check.names = FALSE, encoding = "UTF-8")
X <- as.matrix(niv[, cols]); colnames(X) <- nombres; rownames(X) <- substr(niv$fecha, 1, 7)
base <- sub("_log_diff$|_diff$", "", names(tr))
Xt <- as.matrix(tr[, match(cols, base)]); colnames(Xt) <- nombres; rownames(Xt) <- substr(tr$fecha, 1, 7)
f17 <- function(m) { m <- as.matrix(m); r <- apply(m, 2, function(x) sprintf("%.17g", x)); if (is.null(dim(r))) r <- matrix(r, nrow = nrow(m)); r }
esc <- function(m, f) write.csv(f17(m), file.path(sal, f), row.names = FALSE)
for (via in c("niveles", "transformadas")) {
  M <- if (via == "niveles") X else Xt
  for (nsd in c(0, 3)) {
    a <- aede_r(serie = M, ndim = 3, nw = 60, nsd = nsd, maxcl = 10)
    esc(a$eigenvalues, sprintf("%s_aede%d_valores.csv", via, nsd))
    esc(a$cutpoints$part, sprintf("%s_aede%d_part.csv", via, nsd))
    esc(cbind(a$least_squares_differences), sprintf("%s_aede%d_diff.csv", via, nsd))
    esc(matrix(a$correlationlsq, nrow = dim(a$correlationlsq)[1]), sprintf("%s_aede%d_corlsq.csv", via, nsd))
    esc(matrix(a$correlationsgn, nrow = dim(a$correlationsgn)[1]), sprintf("%s_aede%d_corsgn.csv", via, nsd))
  }
  df <- as.data.frame(M)
  invisible(capture.output(h <- clahfac_r(df, out = TRUE)))
  cr <- clahfacsub_r(df)
  esc(h$hierar, sprintf("%s_hfc_hierar.csv", via)); esc(h$repvar, sprintf("%s_hfc_repvar.csv", via))
  esc(h$corout, sprintf("%s_hfc_corout.csv", via)); esc(h$correp, sprintf("%s_hfc_correp.csv", via))
  esc(h$inertia, sprintf("%s_hfc_inertia.csv", via)); esc(h$nclass, sprintf("%s_hfc_nclass.csv", via))
  esc(cbind(h$hier$order), sprintf("%s_hfc_order.csv", via))
  esc(cbind(cr$eigentab_convergido), sprintf("%s_hfc_eigconv.csv", via))
  esc(cr$datout, sprintf("%s_hfc_datout.csv", via)); esc(cr$npart, sprintf("%s_hfc_npart.csv", via))
  esc(cbind(cr$nsg), sprintf("%s_hfc_nsg.csv", via)); esc(cr$noeuind, sprintf("%s_hfc_noeuind.csv", via))
  items <- do.call(rbind, lapply(1:4, function(k) {
    invisible(capture.output(pc <- clahpart_r(h, noeud = k)))
    v <- unlist(lapply(pc$coord, function(z) c(z$var, z$faccov, z$sup, z$supcov, z$dat, z$ind)))
    esc(cbind(unname(v)), sprintf("%s_hfc_clahpart_coord%d.csv", via, k))
    data.frame(k = k, class = pc$itemclas$class, label = pc$itemclas$label)
  }))
  write.csv(items, file.path(sal, sprintf("%s_hfc_clahpart.csv", via)), row.names = FALSE, fileEncoding = "UTF-8")
  pc4 <- .partclas_ac(h$hier, 4)
  write.csv(data.frame(pos = pc4$partition$Position, ord = pc4$ord.partition$Position),
            file.path(sal, sprintf("%s_hfc_partclas4.csv", via)), row.names = FALSE)
  esc(.hc2axes_ac(h$hier), sprintf("%s_hfc_hc2axes.csv", via))
  ini <- seq(1, nrow(M) - 60 + 1, by = 3)
  jj <- do.call(rbind, lapply(seq_along(ini), function(w) {
    invisible(capture.output(r <- clahfac_r(df[ini[w]:(ini[w] + 59), ], out = TRUE)))
    data.frame(ventana = w, m1 = r$hier$merge[, 1], m2 = r$hier$merge[, 2],
               altura = sprintf("%.17g", r$hier$height), eigentab = sprintf("%.17g", r$hierar[, "Eigentab"]))
  }))
  write.csv(jj, file.path(sal, sprintf("%s_ehfc_ventanas.csv", via)), row.names = FALSE)
}
cat("volcado completo\n")
"""

def calcular_todo():
    # Recalcula con el motor de valores propios activo todas las salidas que se comparan
    out = {}
    for via, (M, d) in VIAS.items():
        r = {'aede': {nsd: cp.aede(M, nw=VENTANA, ndim=NDIM, nsd=nsd, maxcl=MAXCL) for nsd in (0, 3)},
             'hfc': cp.clahfac(M)}
        r['ventanas'] = [cp.clahfac(M.iloc[i:i + VENTANA]) for i in range(0, len(M) - VENTANA + 1, PASO)]
        out[via] = r
    return out

def comparar_directa(res, tmp):
    leer = lambda f: pd.read_csv(tmp / f, float_precision='round_trip').to_numpy(dtype=float)
    REF = 'R/camiz_port.R ejecutado en este entorno'
    an = lambda prueba, via, tipo, dif, detalle='': anotar('Directa · ' + prueba, via, REF, tipo, dif, detalle, grupo='complementaria')
    for via in VIAS:
        for nsd in (0, 3):
            a = res[via]['aede'][nsd]; et = f'aede nsd={nsd}'
            an(f'{et}: valores propios por ventana', via, 'tolerancia', dif_num(leer(f'{via}_aede{nsd}_valores.csv'), a['eigenvalues']))
            part_r = leer(f'{via}_aede{nsd}_part.csv'); part_p = a['cutpoints']['part'].to_numpy()
            an(f'{et}: SC intra de Fisher', via, 'tolerancia', dif_num(part_r[:, 1], part_p[:, 1]))
            an(f'{et}: Calinski-Harabász (k gl)', via, 'tolerancia', dif_num(part_r[:, 2], part_p[:, 2]),
               f'máximo del índice: {np.nanmax(part_r[:, 2]):.1f}')
            an(f'{et}: límites de clase v_1..v_10', via, 'exacta', int((part_r[:, 3:] != part_p[:, 3:]).sum()))
            wn = a['eigenvalues'].shape[0]
            for nom, corto in [('correlationsgn', 'corsgn'), ('correlationlsq', 'corlsq')]:
                an(f'{et}: {nom}', via, 'tolerancia', dif_num(leer(f'{via}_aede{nsd}_{corto}.csv'), a[nom].reshape(wn, -1, order='F')))
            an(f'{et}: diferencias de mínimos cuadrados', via, 'tolerancia',
               dif_num(leer(f'{via}_aede{nsd}_diff.csv')[:, 0], a['least_squares_differences']))
        h = res[via]['hfc']; cr = h['crudo']
        an('clahfac: jerarquía completa (hierar)', via, 'tolerancia', dif_num(leer(f'{via}_hfc_hierar.csv'), h['hierar']))
        for nom, val in [('repvar', h['repvar']), ('corout', h['corout']), ('correp', h['correp']),
                         ('inertia', h['inertia']), ('datout', cr['datout'])]:
            an(f'clahfac: {nom}', via, 'tolerancia', dif_num(leer(f'{via}_hfc_{nom}.csv'), val))
        an('clahfac: Eigentab convergido', via, 'tolerancia', dif_num(leer(f'{via}_hfc_eigconv.csv')[:, 0], cr['eigentab_convergido']))
        for nom, val in [('nclass', h['nclass']), ('npart', cr['npart']), ('noeuind', cr['noeuind'])]:
            an(f'clahfac: {nom}', via, 'exacta', int((leer(f'{via}_hfc_{nom}.csv') != np.asarray(val)).sum()))
        an('clahfac: orden del dendrograma y nsg', via, 'exacta',
           int((leer(f'{via}_hfc_order.csv')[:, 0] != h['hier']['order']).sum() + (leer(f'{via}_hfc_nsg.csv')[:, 0] != cr['nsg']).sum()))
        it_r = pd.read_csv(tmp / f'{via}_hfc_clahpart.csv', encoding='utf-8')
        n_dif, dif_c = 0, 0.0
        for k in range(1, 5):
            pk = cp.clahpart(h, noeud=k); ik = it_r[it_r.k == k]
            n_dif += int(len(ik) != len(pk['itemclas'])) or int(((ik['class'].astype(str).to_numpy() != pk['itemclas']['class'].to_numpy()) |
                                                                  (ik['label'].to_numpy() != pk['itemclas']['label'].to_numpy())).sum())
            partes = []
            for z in pk['coord']:
                partes += [np.ravel(z['var'], order='F'), z['faccov']]
                if z['sup'] is not None:
                    partes += [np.ravel(z['sup'], order='F'), np.ravel(z['supcov'], order='F')]
                partes += [np.ravel(z['dat'], order='F'), np.ravel(z['ind'], order='F')]
            dif_c = max(dif_c, dif_num(leer(f'{via}_hfc_clahpart_coord{k}.csv')[:, 0], np.concatenate(partes)))
        an('clahpart k=1..4: clases e ítems', via, 'exacta', n_dif)
        an('clahpart k=1..4: coordenadas', via, 'tolerancia', dif_c)
        pc4 = cp.partclas(h['hier'], 4); p4r = leer(f'{via}_hfc_partclas4.csv')
        an('partclas (4 clases)', via, 'exacta',
           int((p4r[:, 0] != pc4['partition']['Position'].to_numpy()).sum() + (p4r[:, 1] != pc4['ord.partition']['Position'].to_numpy()).sum()))
        an('hc2axes', via, 'tolerancia', dif_num(leer(f'{via}_hfc_hc2axes.csv'), cp.hc2axes(h['hier'])))
        jj = pd.read_csv(tmp / f'{via}_ehfc_ventanas.csv', float_precision='round_trip')
        merges = np.vstack([v['hier']['merge'] for v in res[via]['ventanas']])
        alturas = np.concatenate([v['hier']['height'] for v in res[via]['ventanas']])
        eigt = np.concatenate([v['hierar']['Eigentab'].to_numpy() for v in res[via]['ventanas']])
        an('HFC por ventanas: fusiones (merge) de todas las ventanas', via, 'exacta',
           int(len(jj) != len(merges)) or int((jj[['m1', 'm2']].to_numpy() != merges).sum()), f'{jj.ventana.max()} ventanas')
        an('HFC por ventanas: alturas y Eigentab', via, 'tolerancia', max(dif_num(jj.altura, alturas), dif_num(jj.eigentab, eigt)))

rscript = shutil.which('Rscript')
if rscript is None:
    print('Rscript no disponible: se omite 7.2 (las pruebas de aceptación no dependen de ella).')
else:
    tmp = Path(tempfile.mkdtemp(prefix='port_camiz_'))
    (tmp / 'volcado.R').write_text(R_VOLCADO, encoding='utf-8')
    salida = subprocess.run([rscript, str(tmp / 'volcado.R'), str(RAIZ), str(tmp), '|'.join(META.columna),
                             '|'.join(META.nombre_corto)], capture_output=True, text=True, timeout=1200)
    assert salida.returncode == 0, salida.stderr[-2000:]
    MOTOR = 'numpy'
    comparar_directa(calcular_todo(), tmp)
    if cp.activar_lapack_referencia():
        MOTOR = 'LAPACK de referencia (dsyevr)'
        try:
            comparar_directa(calcular_todo(), tmp)
        finally:
            cp.desactivar_lapack_referencia()
            MOTOR = 'numpy'
    else:
        print('No se encontró el LAPACK de referencia: no se puede aislar el efecto de la rutina de valores propios.')
    shutil.rmtree(tmp)

# Explicación de las diferencias con numpy que desaparecen con el LAPACK de R
reg = pd.DataFrame(registro)
for i, fila in reg.iterrows():
    if fila.estado == 'difiere' and fila.motor_valores_propios == 'numpy' and fila.grupo == 'complementaria':
        par = reg[(reg.prueba == fila.prueba) & (reg.via == fila.via) & (reg.motor_valores_propios != 'numpy')]
        if len(par) and (par.estado == 'coincide').all():
            registro[i]['explicacion'] = (
                'Rutina de valores propios: numpy (dsyevd sobre OpenBLAS) difiere de dsyevr del LAPACK de referencia '
                'que usa eigen() de R en los últimos bits; con el mismo LAPACK la prueba coincide '
                f"(dif. {par.dif_max.max():.3g}). La SC intra de Fisher resta cantidades grandes y casi iguales "
                'y amplifica esos bits.')
pd.DataFrame(registro).query("grupo == 'complementaria'")[['prueba', 'via', 'motor_valores_propios', 'dif_max', 'estado']]


,prueba,via,motor_valores_propios,dif_max,estado
32,Lector pandas · aede: k y cortes,niveles,numpy,0.000000e+00,coincide
33,Lector pandas · HFC: N1/N2,niveles,numpy,0.000000e+00,coincide
34,Lector pandas · HFC: cifras de la jerarquía,niveles,numpy,4.973799e-14,coincide
35,Lector pandas · aede: k y cortes,transformadas,numpy,0.000000e+00,coincide
36,Lector pandas · HFC: N1/N2,transformadas,numpy,0.000000e+00,coincide
...,...,...,...,...,...
157,Directa · clahpart k=1..4: coordenadas,transformadas,LAPACK de referencia (dsyevr),1.110223e-16,coincide
158,Directa · partclas (4 clases),transformadas,LAPACK de referencia (dsyevr),0.000000e+00,coincide
159,Directa · hc2axes,transformadas,LAPACK de referencia (dsyevr),0.000000e+00,coincide
160,Directa · HFC por ventanas: fusiones (merge) d...,transformadas,LAPACK de referencia (dsyevr),0.000000e+00,coincide


## 8. Resultado de la paridad

In [10]:
paridad = pd.DataFrame(registro)
paridad.to_csv(DIR_T / '999_port_paridad.csv', index=False)
with pd.option_context('display.max_rows', 200, 'display.max_colwidth', 70):
    display(paridad[['grupo', 'prueba', 'via', 'motor_valores_propios', 'tipo', 'dif_max', 'estado']])
print(f"{(paridad.estado == 'coincide').sum()} de {len(paridad)} pruebas coinciden")
# Criterio: todas las pruebas de aceptación coinciden; en las complementarias, toda diferencia
# debe quedar explicada por una prueba que la aísle (no se amplía ninguna tolerancia)
acept = paridad[paridad.grupo == 'aceptación']
assert (acept.estado == 'coincide').all(), 'Pruebas de aceptación que difieren:\n' + acept[acept.estado != 'coincide'].to_string()
sin_expl = paridad[(paridad.estado != 'coincide') & (paridad.explicacion == '')]
assert sin_expl.empty, 'Diferencias sin explicar:\n' + sin_expl.to_string()
print('Aceptación:', int((acept.estado == 'coincide').sum()), 'de', len(acept),
      '· complementarias que difieren (todas explicadas):', int((paridad.estado != 'coincide').sum()))


,grupo,prueba,via,motor_valores_propios,tipo,dif_max,estado
0,aceptación,aede: k elegido (6 criterios),niveles,numpy,exacta,0.000000e+00,coincide
1,aceptación,aede: fechas de corte (6 criterios),niveles,numpy,exacta,0.000000e+00,coincide
2,aceptación,aede: k elegido (6 criterios),transformadas,numpy,exacta,0.000000e+00,coincide
3,aceptación,aede: fechas de corte (6 criterios),transformadas,numpy,exacta,0.000000e+00,coincide
4,aceptación,HFC: nodos unidos N1/N2 (por código de serie),niveles,numpy,exacta,0.000000e+00,coincide
5,aceptación,HFC: nombres de las series unidas,niveles,numpy,exacta,0.000000e+00,coincide
6,aceptación,HFC: tamaño de cada nodo (Num),niveles,numpy,exacta,0.000000e+00,coincide
7,aceptación,"HFC: λ1 e índice (Eigenv, Index)",niveles,numpy,tolerancia,2.664535e-14,coincide
8,aceptación,HFC: porcentajes locales y globales,niveles,numpy,tolerancia,4.973799e-14,coincide
9,aceptación,HFC: Eigentab y Ratio,niveles,numpy,tolerancia,4.796163e-14,coincide


160 de 162 pruebas coinciden
Aceptación: 32 de 32 · complementarias que difieren (todas explicadas): 2


**Lectura.** Si todas las pruebas coinciden, el port de Python reproduce la versión en R
(referencia, D6) en las dos vías: los mismos $k$, los mismos cortes de Fisher, la misma jerarquía
de la HFC por código de serie y las mismas cifras hasta el redondeo de punto flotante. Las
diferencias residuales (del orden de $10^{-14}$) provienen de la escritura de los CSV con 15
cifras significativas y del orden de las operaciones en BLAS/LAPACK, no del método.